### Phase 3: Causal Estimation

In this phase, the causal effect of the treatment variable on the outcome variable is estimated using a causal inference model.

The Directed Acyclic Graph (DAG) constructed in the previous phase is used as the structural input for the model.

The estimation is performed using the DoWhy framework.

In [2]:
import pandas as pd
import numpy as np
import pickle
from dowhy import CausalModel
print("Everything Loaded Successfully")

Everything Loaded Successfully


In [20]:
student_df = pd.read_csv("../data/raw/student-mat.csv", sep = ";")
print("Data loaded:", student_df.shape)

Data loaded: (395, 33)


In [27]:
with open("../artifacts/phase1/data_schema.pkl", "rb") as f:
    SCHEMA = pickle.load(f)
print("Schema loaded:", SCHEMA)

Schema loaded: {'treatment': 'studytime', 'outcome': 'G3', 'confounders': ['absences', 'health', 'failures'], 'features': ['studytime', 'absences', 'health', 'failures']}


In [28]:
with open("../artifacts/phase2/causal_dag.pkl", "rb") as f:
    causal_graph = pickle.load(f)
print("Causal Graph:")
print(causal_graph)

Causal Graph:
digraph {
    studytime -> G3;
    absences -> studytime;
    absences -> G3;
    health -> studytime;
    health -> G3;
    failures -> studytime;
    failures -> G3;
}


### Loading Causal Graph

The DAG constructed in the previous phase is loaded and used to define the causal structure required for estimation.

In [29]:
model = CausalModel(data = student_df, treatment = SCHEMA["treatment"], outcome = SCHEMA["outcome"], graph = causal_graph)
print("Causal model created successfully")

Causal model created successfully


### Causal Model Construction

A causal model is created using the dataset, treatment variable, outcome variable, and the DAG.

This model forms the basis for identifying and estimating the causal effect.

In [30]:
identified_estimand = model.identify_effect()
print("Identified Estimand:")
print(identified_estimand)

Identified Estimand:
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                                      
────────────(E[G3|health,failures,absences])
d[studytime]                                
Estimand assumption 1, Unconfoundedness: If U→{studytime} and U→G3 then P(G3|studytime,health,failures,absences,U) = P(G3|studytime,health,failures,absences)

### Estimand : 2
Estimand name: iv
No such variable(s) found!

### Estimand : 3
Estimand name: frontdoor
No such variable(s) found!

### Estimand : 4
Estimand name: general_adjustment
Estimand expression:
     d                                      
────────────(E[G3|health,failures,absences])
d[studytime]                                
Estimand assumption 1, Unconfoundedness: If U→{studytime} and U→G3 then P(G3|studytime,health,failures,absences,U) = P(G3|studytime,health,failures,absences)



### Effect Identification

The causal effect is identified using the model.

This step determines how the causal effect can be estimated based on the defined graph structure.

In [31]:
estimate = model.estimate_effect(identified_estimand, method_name = "backdoor.linear_regression")
print("Causal Effect Estimate:")
print(estimate)

Causal Effect Estimate:
*** Causal Estimate ***

## Identified estimand
Estimand type: EstimandType.NONPARAMETRIC_ATE

### Estimand : 1
Estimand name: backdoor
Estimand expression:
     d                                      
────────────(E[G3|health,failures,absences])
d[studytime]                                
Estimand assumption 1, Unconfoundedness: If U→{studytime} and U→G3 then P(G3|studytime,health,failures,absences,U) = P(G3|studytime,health,failures,absences)

## Realized estimand
b: G3~studytime+health+failures+absences
Target units: ate

## Estimate
Mean value: 0.20336046135752106



### Effect Estimation

The Average Treatment Effect (ATE) is estimated using a linear regression method.

This provides a numerical value representing the impact of the treatment variable on the outcome variable.

In [32]:
print("Final Effect Value:", estimate.value)

Final Effect Value: 0.20336046135752106


### Estimated Effect

The printed value represents the estimated causal effect of study time on the final grade.

This value indicates how changes in the treatment variable influence the outcome.

In [33]:
with open("../artifacts/phase3/causal_model.pkl", "wb") as f:
    pickle.dump(model, f)
with open("../artifacts/phase3/causal_effect.pkl", "wb") as f:
    pickle.dump(estimate, f)

### Saving Causal Model

The trained causal model is saved for reuse in later phases.

This ensures consistency across the pipeline.

### Saving Estimated Effect

The computed causal effect is stored for use in subsequent phases such as counterfactual analysis and evaluation.

### Conclusion

This phase successfully estimates the causal effect between the treatment and outcome variables using the defined DAG and causal model.

The result provides a baseline effect that will be compared with counterfactual and uncertainty-based estimates in later phases.